In [ ]:
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# 1. 大規模データの生成 (10万サンプル、20特徴量)
# ==========================================
print("サンプルデータを生成中...")
X, y = make_classification(
    n_samples=100000,   # 10万サンプル
    n_features=20,     # 20個の特徴量
    n_informative=15,  # 予測に意味のある特徴量
    random_state=42
)

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"訓練データサイズ: {X_train.shape}")
print(f"テストデータサイズ: {X_test.shape}\n" + "-"*40)


# ==========================================
# 2. 事前剪定 (Pre-pruning) の実行
# ==========================================
start_time = time.time()

# 浅い木に制限して一気に学習
pre_model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=42)
pre_model.fit(X_train, y_train)

pre_time = time.time() - start_time
pre_acc = accuracy_score(y_test, pre_model.predict(X_test))

print("【事前剪定】")
print(f"計算時間: {pre_time:.4f} 秒")
print(f"最大深さ: {pre_model.get_depth()}")
print(f"テストデータ正解率: {pre_acc:.4f}\n" + "-"*40)


# ==========================================
# 3. 事後剪定 (Post-pruning) の実行
# ==========================================
# ※大規模データですべてのalphaを愚直にループするとフリーズする恐れがあるため、
#   効率的なステップ（間引き）で alpha を探索します。
start_time = time.time()

# ステップA: まず最大まで育てる
full_model = DecisionTreeClassifier(random_state=42)
full_model.fit(X_train, y_train)

# ステップB: 剪定パス（alpha候補）の取得
path = full_model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# ※大規模データだとalphaの候補が数千〜数万個になるため、等間隔に10個だけピックアップして高速化
idx = np.linspace(0, len(ccp_alphas) - 1, 10, dtype=int)
subset_alphas = ccp_alphas[idx]

# ステップC: 限定したalphaの中でベストを探す
best_acc = 0
best_alpha = 0

for alpha in subset_alphas:
    # 既存の木を削るのではなく再学習が必要なため、ここでも時間がかかります
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    if acc > best_acc:
        best_acc = acc
        best_alpha = alpha

post_time = time.time() - start_time

print("【事後剪定】")
print(f"計算時間: {post_time:.4f} 秒 ")
print(f"選択されたハイパーパラメータ: {best_alpha:.6f}")
print(f"テストデータ正解率: {best_acc:.4f}\n" + "-"*40)

# ==========================================
# 結論の出力
# ==========================================
print(f"計算時間の差: 事後剪定は事前剪定の 【{post_time / pre_time:.1f}倍】 の時間がかかりました。")



In [ ]:
# ==========================================
# 1. 小規模データの生成 (1000サンプル、20特徴量)
# ==========================================
print("サンプルデータを生成中...")
X, y = make_classification(
    n_samples=1000,   # 1000サンプル
    n_features=20,     # 20個の特徴量
    n_informative=15,  # 予測に意味のある特徴量
    random_state=42
)

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"訓練データサイズ: {X_train.shape}")
print(f"テストデータサイズ: {X_test.shape}\n" + "-"*40)


# ==========================================
# 2. 事前剪定 (Pre-pruning) の実行
# ==========================================
start_time = time.time()

# 浅い木に制限して一気に学習
pre_model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=42)
pre_model.fit(X_train, y_train)

pre_time = time.time() - start_time
pre_acc = accuracy_score(y_test, pre_model.predict(X_test))

print("【事前剪定】")
print(f"計算時間: {pre_time:.4f} 秒")
print(f"最大深さ: {pre_model.get_depth()}")
print(f"テストデータ正解率: {pre_acc:.4f}\n" + "-"*40)


# ==========================================
# 3. 事後剪定 (Post-pruning) の実行
# ==========================================
# ※大規模データですべてのalphaを愚直にループするとフリーズする恐れがあるため、
#   効率的なステップ（間引き）で alpha を探索します。
start_time = time.time()

# ステップA: まず最大まで育てる
full_model = DecisionTreeClassifier(random_state=42)
full_model.fit(X_train, y_train)

# ステップB: 剪定パス（alpha候補）の取得
path = full_model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# ※大規模データだとalphaの候補が数千〜数万個になるため、等間隔に10個だけピックアップして高速化
idx = np.linspace(0, len(ccp_alphas) - 1, 10, dtype=int)
subset_alphas = ccp_alphas[idx]

# ステップC: 限定したalphaの中でベストを探す
best_acc = 0
best_alpha = 0

for alpha in subset_alphas:
    # 既存の木を削るのではなく再学習が必要なため、ここでも時間がかかります
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    if acc > best_acc:
        best_acc = acc
        best_alpha = alpha

post_time = time.time() - start_time

print("【事後剪定】")
print(f"計算時間: {post_time:.4f} 秒")
print(f"選択された最適なハイパーパラメータ: {best_alpha:.6f}")
print(f"テストデータ正解率: {best_acc:.4f}\n" + "-"*40)

# ==========================================
# 結論の出力
# ==========================================
print(f"計算時間の差: 事後剪定は事前剪定の 【{post_time / pre_time:.1f}倍】 の時間がかかりました。")


In [ ]:
# ==========================================
# 1. 実験の設定
# ==========================================
# 1,000 から 100,000 まで、10,000刻みでデータ件数を定義
sample_sizes = [1000] + list(range(10000, 100001, 10000))

pre_times = []
post_times = []

print("実験を開始します。データ件数ごとの計算時間を計測中...")

# ==========================================
# 2. 各データサイズでの時間計測ループ
# ==========================================
for size in sample_sizes:
    # データの生成
    X, y = make_classification(
        n_samples=size,
        n_features=20,
        n_informative=15,
        random_state=42
    )

    # --- A. 事前剪定 (Pre-pruning) の計測 ---
    start_pre = time.time()
    pre_model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=42)
    pre_model.fit(X, y)
    pre_times.append(time.time() - start_pre)

    # --- B. 事後剪定 (Post-pruning) の計測 ---
    start_post = time.time()
    # 1. 最大まで育てる
    full_model = DecisionTreeClassifier(random_state=42)
    full_model.fit(X, y)
    # 2. アルファの候補を取得
    path = full_model.cost_complexity_pruning_path(X, y)
    ccp_alphas = path.ccp_alphas
    # 3. 10個のアルファに絞って再学習コストを計測 (大規模データでのフリーズ防止)
    idx = np.linspace(0, len(ccp_alphas) - 1, 10, dtype=int)
    subset_alphas = ccp_alphas[idx]

    for alpha in subset_alphas:
        clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
        clf.fit(X, y)

    post_times.append(time.time() - start_post)
    print(f"データ件数: {size:>7,} 件 完了 (事前: {pre_times[-1]:.4f}秒 / 事後: {post_times[-1]:.4f}秒)")

print("\nすべて完了しました。グラフを描画します。")

# ==========================================
# 3. 折れ線グラフのプロット
# ==========================================
plt.figure(figsize=(10, 6), dpi=150)

# 日本語フォントが化ける場合は英語表記にしています
plt.plot(sample_sizes, pre_times, marker='o', color='#1f77b4', linewidth=2.5, label='Pre-pruning (事前剪定)')
plt.plot(sample_sizes, post_times, marker='s', color='#9467bd', linewidth=2.5, label='Post-pruning (事後剪定)')

# グラフのデザイン調整
plt.title('Computation Time Comparison by Data Size', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Samples (データ件数)', fontsize=12)
plt.ylabel('Execution Time (計算時間 [秒])', fontsize=12)
plt.xticks(sample_sizes, [f"{int(s/1000)}k" if s>=10000 else "1k" for s in sample_sizes], rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12, loc='upper left')

# レイアウトの自動調整と保存
plt.tight_layout()
plt.savefig('pruning_time_comparison.png') # スライド貼付用に画像保存
plt.show()